In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import GridSearchCV


In [2]:
# Load the dataset
df = pd.read_csv('unsegmented_features.csv')
df.head()

,x_min,y_min,x_max,y_max,width,height,diagonal_length,fish_area,image_area,area_ratio,...,centroid_x,centroid_y,norm_centroid_x,norm_centroid_y,center_x,center_y,distance_to_center,perimeter,compactness,length_cm
0,956.64280,362.55695,1407.2202,2207.4739,450.5774,1844.91695,1899.141529,831277.882547,5992704,0.138715,...,1181.93150,1285.015425,0.643754,0.393693,918.0,1632.0,435.956571,4590.98870,0.039440,13.0
1,1105.53050,959.79390,2940.9062,1348.7906,388.9967,1835.37570,1876.145621,713955.090560,5992704,0.119137,...,2023.21835,1154.292250,0.619859,0.628699,1632.0,918.0,457.040288,4448.74480,0.036074,14.0
2,989.95340,1060.81400,2807.6096,1455.5302,394.7162,1817.65620,1860.020145,717458.348170,5992704,0.119722,...,1898.78150,1258.172100,0.581735,0.685279,1632.0,918.0,432.307097,4424.74480,0.036645,12.5
3,1152.86270,1149.84680,2883.3828,1587.5240,437.6772,1730.52010,1785.010125,757409.191912,5992704,0.126389,...,2018.12275,1368.685400,0.618297,0.745471,1632.0,918.0,593.471236,4336.39460,0.040278,13.0
4,748.52936,1007.09030,2756.3474,1423.0254,415.9351,2007.81804,2050.447583,835121.997249,5992704,0.139356,...,1752.43838,1215.057850,0.536899,0.661796,1632.0,918.0,320.544489,4847.50628,0.035540,13.5


In [3]:
# Custom transformer to drop null values
class DropNullValues(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        # Convert to DataFrame if needed, drop nulls, and return as NumPy array
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X)
        return X.dropna().values

# Custom transformer to drop features with low or zero correlation
class DropLowCorrelationFeatures(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.005):
        self.threshold = threshold
        self.features_to_keep = []

    def fit(self, X, y=None):
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X)
        corr_matrix = X.corrwith(y).abs()
        self.features_to_keep = corr_matrix[corr_matrix > self.threshold].index.tolist()
        return self

    def transform(self, X):
        if isinstance(X, np.ndarray):
            X = pd.DataFrame(X)
        return X[self.features_to_keep].values


In [4]:
# Function to create a pipeline with all steps included
def create_pipeline(model):
    pipeline = Pipeline([
        ('drop_nulls', DropNullValues()),
        ('drop_low_corr', DropLowCorrelationFeatures(threshold=0.005)),
        ('scaler', StandardScaler()),
        ('model', model)
    ])
    return pipeline

# Function to evaluate models
def evaluate_models(models, X_train, y_train, X_test, y_test):
    results = {}
    
    for name, model in models:
        # Train the model
        model.fit(X_train, y_train)
        
        # Predict on the test set
        y_pred = model.predict(X_test)
        
        # Calculate performance metrics
        mse = mean_squared_error(y_test, y_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_test, y_pred)
        
        # Store the results
        results[name] = {
            'RMSE': rmse,
            'R^2': r2
        }
    
    return results


In [5]:
# Define features and target variable
X = df.drop(columns=['length_cm', 'Unnamed: 0'], errors='ignore')
y = df['length_cm']

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=28)


In [6]:
# Define models with the new pipeline steps included
models_with_pipeline = [
    ('Linear Regression', create_pipeline(LinearRegression())),
    ('Ridge Regression', create_pipeline(Ridge(random_state=28))),
    ('Lasso Regression', create_pipeline(Lasso(random_state=28))),
    ('Random Forest', create_pipeline(RandomForestRegressor(n_estimators=50, random_state=28))),
    ('Support Vector Regressor', create_pipeline(SVR())),
    ('K-Nearest Neighbors', create_pipeline(KNeighborsRegressor(n_neighbors=5))),
    ('Decision Tree', create_pipeline(DecisionTreeRegressor(random_state=28))),
    ('Gradient Boosting', create_pipeline(GradientBoostingRegressor(n_estimators=50, random_state=28))),
    ('AdaBoost', create_pipeline(AdaBoostRegressor(n_estimators=50, random_state=28))),
    ('XGBoost', create_pipeline(XGBRegressor(n_estimators=50, random_state=28, objective='reg:squarederror')))
]


In [7]:
# Evaluate the models
model_results = evaluate_models(models_with_pipeline, X_train, y_train, X_test, y_test)

In [8]:
model_results_df = pd.DataFrame(model_results).T
model_results_df

,RMSE,R^2
Linear Regression,0.844281,0.227582
Ridge Regression,0.823210,0.265656
Lasso Regression,0.966800,-0.012865
Random Forest,0.813411,0.283034
Support Vector Regressor,0.872510,0.175066
K-Nearest Neighbors,0.799637,0.307111
Decision Tree,1.056541,-0.209626
Gradient Boosting,0.833933,0.246401
AdaBoost,0.810602,0.287978
XGBoost,0.890670,0.140369


In [9]:
# Define the parameter grid for AdaBoost
adaboost_param_grid = {
    'model__n_estimators': [50, 100, 200],
    'model__learning_rate': [0.01, 0.1, 1],
    'model__loss': ['linear', 'square', 'exponential']
}

# Define the parameter grid for Ridge Regression
ridge_param_grid = {
    'model__alpha': [0.01, 0.1, 1, 10, 100],
}

# Define the parameter grid for Gradient Boosting
gb_param_grid = {
    'model__n_estimators': [50, 100, 200],
    'model__learning_rate': [0.01, 0.1, 0.5],
    'model__max_depth': [3, 5, 7],
}

# Define the parameter grid for SVR
svr_param_grid = {
    'model__C': [0.1, 1, 10],
    'model__epsilon': [0.01, 0.1, 1],
    'model__kernel': ['linear', 'rbf', 'poly']
}

# Define the parameter grid for XGBoost
xgb_param_grid = {
    'model__n_estimators': [50, 100, 200],
    'model__learning_rate': [0.01, 0.1, 0.2],
    'model__max_depth': [3, 5, 7],
    'model__subsample': [0.8, 1.0],
    'model__colsample_bytree': [0.8, 1.0]
}

# Define the parameter grid for Random Forest
rf_param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
}


In [2]:
# install PyPDF2
! pip install PyPDF2

     ---------------------------------------- 0.0/232.6 kB ? eta -:--:--
     ----- --------------------------------- 30.7/232.6 kB 1.4 MB/s eta 0:00:01
     --------------- ----------------------- 92.2/232.6 kB 1.3 MB/s eta 0:00:01
     -------------------------------------- 232.6/232.6 kB 1.8 MB/s eta 0:00:00


In [10]:
# Create pipelines for each model
adaboost_pipeline = create_pipeline(AdaBoostRegressor(random_state=28))
ridge_pipeline = create_pipeline(Ridge(random_state=28))
gb_pipeline = create_pipeline(GradientBoostingRegressor(random_state=28))
svr_pipeline = create_pipeline(SVR())
xgb_pipeline = create_pipeline(XGBRegressor(random_state=28))
rf_pipeline = create_pipeline(RandomForestRegressor(random_state=28))

# Perform Grid Search for SVR
svr_grid_search = GridSearchCV(svr_pipeline, svr_param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
svr_grid_search.fit(X_train, y_train)

# Perform Grid Search for XGBoost
xgb_grid_search = GridSearchCV(xgb_pipeline, xgb_param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
xgb_grid_search.fit(X_train, y_train)

# Perform Grid Search for Random Forest
rf_grid_search = GridSearchCV(rf_pipeline, rf_param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
rf_grid_search.fit(X_train, y_train)


# Perform Grid Search for AdaBoost
adaboost_grid_search = GridSearchCV(adaboost_pipeline, adaboost_param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
adaboost_grid_search.fit(X_train, y_train)

# Perform Grid Search for Ridge Regression
ridge_grid_search = GridSearchCV(ridge_pipeline, ridge_param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
ridge_grid_search.fit(X_train, y_train)

# Perform Grid Search for Gradient Boosting
gb_grid_search = GridSearchCV(gb_pipeline, gb_param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
gb_grid_search.fit(X_train, y_train)


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('drop_nulls', DropNullValues()),
                                       ('drop_low_corr',
                                        DropLowCorrelationFeatures()),
                                       ('scaler', StandardScaler()),
                                       ('model',
                                        GradientBoostingRegressor(random_state=28))]),
             n_jobs=-1,
             param_grid={'model__learning_rate': [0.01, 0.1, 0.5],
                         'model__max_depth': [3, 5, 7],
                         'model__n_estimators': [50, 100, 200]},
             scoring='neg_mean_squared_error')

In [11]:
# Best models from Grid Search
best_adaboost = adaboost_grid_search.best_estimator_
best_ridge = ridge_grid_search.best_estimator_
best_gb = gb_grid_search.best_estimator_
best_svr = svr_grid_search.best_estimator_
best_xgb = xgb_grid_search.best_estimator_
best_rf = rf_grid_search.best_estimator_

# Evaluate the best models on the test set
models_best = [
    ('Best AdaBoost', best_adaboost),
    ('Best Ridge Regression', best_ridge),
    ('Best Gradient Boosting', best_gb),
    ('Best SVR', best_svr),
    ('Best XGBoost', best_xgb),
    ('Best Random Forest', best_rf)
]

best_model_results = evaluate_models(models_best, X_test, y_test, X_test, y_test)
best_model_results_df = pd.DataFrame(best_model_results).T

# Display the results
best_model_results_df

,RMSE,R^2
Best AdaBoost,0.491499,0.738228
Best Ridge Regression,0.731922,0.419492
Best Gradient Boosting,0.463144,0.767560
Best SVR,0.719635,0.438819
Best XGBoost,0.514115,0.713582
Best Random Forest,0.549469,0.672836


In [12]:
# Best hyperparameters for each model
print("Best hyperparameters for AdaBoost:")
print(adaboost_grid_search.best_params_)

print("\nBest hyperparameters for Ridge Regression:")
print(ridge_grid_search.best_params_)

print("\nBest hyperparameters for Gradient Boosting:")
print(gb_grid_search.best_params_)

# Best hyperparameters for each model
print("Best hyperparameters for SVR:")
print(svr_grid_search.best_params_)

print("\nBest hyperparameters for XGBoost:")
print(xgb_grid_search.best_params_)

print("\nBest hyperparameters for Random Forest:")
print(rf_grid_search.best_params_)


Best hyperparameters for AdaBoost:
{'model__learning_rate': 0.1, 'model__loss': 'exponential', 'model__n_estimators': 200}

Best hyperparameters for Ridge Regression:
{'model__alpha': 10}

Best hyperparameters for Gradient Boosting:
{'model__learning_rate': 0.01, 'model__max_depth': 3, 'model__n_estimators': 200}
Best hyperparameters for SVR:
{'model__C': 1, 'model__epsilon': 0.01, 'model__kernel': 'linear'}

Best hyperparameters for XGBoost:
{'model__colsample_bytree': 1.0, 'model__learning_rate': 0.01, 'model__max_depth': 3, 'model__n_estimators': 200, 'model__subsample': 0.8}

Best hyperparameters for Random Forest:
{'model__max_depth': 10, 'model__min_samples_leaf': 4, 'model__min_samples_split': 10, 'model__n_estimators': 300}


In [17]:
import pickle

# Save the best AdaBoost model to a file
with open('best_adaboost_model(unsegmented).pkl', 'wb') as model_file:
    pickle.dump(best_adaboost, model_file)


In [22]:
with open('best_adaboost_model(unsegmented).pkl', 'rb') as model_file:
    loaded_adaboost_model = pickle.load(model_file)
    
# Predict on the test set
y_pred = loaded_adaboost_model.predict(X_test)

# Create a dataframe with true values, predicted values, and absolute differences
results_df = pd.DataFrame({
    'Y_true': y_test,
    'Y_pred': y_pred,
    'Absolute_Difference': abs(y_test - y_pred)
})

# Display the results dataframe
results_df.head()


,Y_true,Y_pred,Absolute_Difference
158,12.5,12.666667,0.166667
88,14.5,14.128205,0.371795
409,12.0,12.227273,0.227273
315,13.0,13.600000,0.600000
385,13.0,12.905660,0.094340


In [23]:
results_df.to_csv('results_adaboost_unsegmented.csv', index=False)

In [24]:
# Calculate statistical variations of the error (absolute difference)
mean_error = results_df['Absolute_Difference'].mean()
max_error = results_df['Absolute_Difference'].max()
min_error = results_df['Absolute_Difference'].min()
median_error = results_df['Absolute_Difference'].median()
mode_error = results_df['Absolute_Difference'].mode()[0]

# Print the results
print(f"Mean Absolute Error: {mean_error:.4f}")
print(f"Max Absolute Error: {max_error:.4f}")
print(f"Min Absolute Error: {min_error:.4f}")
print(f"Median Absolute Error: {median_error:.4f}")
print(f"Mode of Absolute Error: {mode_error:.4f}")


Mean Absolute Error: 0.4108
Max Absolute Error: 0.9091
Min Absolute Error: 0.0000
Median Absolute Error: 0.3722
Mode of Absolute Error: 0.0000
